<a href="https://colab.research.google.com/github/banwancha/AutoMergePublicNodes/blob/master/index_tts2_api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **1.安装 IndexTTS-2**

In [ ]:
%cd /content
# !rm -rf /content/index-tts
!git lfs install
!git clone https://github.com/secpo/index-tts.git
%cd index-tts
!git lfs pull
!wget https://raw.githubusercontent.com/NeuralFalconYT/Useful-Function/refs/heads/main/hf_downloader.py
!pip install uv --quiet
!uv sync --all-extras
from IPython.display import clear_output
clear_output()

### **2.下载模型**

In [ ]:
%cd /content/index-tts
from hf_downloader import download_model
from IPython.display import clear_output
def add_share(file_path="/content/index-tts/webui.py"):
    with open(file_path, "r") as f:
        lines = f.readlines()

    for i, line in enumerate(lines):
        if "demo.launch" in line:
            lines[i] = "    demo.launch(share=True, debug=True)\n"

    with open(file_path, "w") as f:
        f.writelines(lines)

    print(f"✅ Updated {file_path} with share=True, debug=True")

model_path = download_model(
    "IndexTeam/IndexTTS-2",
    download_folder="./checkpoints",
    redownload=False
)
clear_output()
print("✅ Model saved at:", model_path)
add_share()

### **3.连接/挂载硬盘**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### **4.启动 API 服务**
启动 webui 运行命令 !uv run webui.py --model_dir "/content/index-tts/checkpoints/IndexTTS-2"

In [ ]:
'''
%cd /content/index-tts
%env INDEXTTS_MODEL_DIR=/content/index-tts/checkpoints/IndexTTS-2
%env INDEXTTS_CFG_PATH=/content/index-tts/checkpoints/IndexTTS-2/config.yaml
%env NGROK_AUTHTOKEN=37DzqJSNqhLTF2YftYaXFSD5gkl_g89szrUYEr7z9nvQKbwo
%env CHUNK_LENGTH=250
%env OUTPUT_DIR=/content/drive/MyDrive/IndexTTS/outputs
!uv run index_tts_api.py --model_dir "/content/index-tts/checkpoints/IndexTTS-2"
'''

In [ ]:
# ---------------------------------------------------------------
# 终极修正方案：IndexTTS 专用版 (锁定端口 7890)
# ---------------------------------------------------------------
import subprocess
import time
import os
import re

# 1. 确保进入正确目录
%cd /content/index-tts

# 2. 杀死旧进程，防止端口被占用
print("🧹 正在清理旧进程...")
!pkill -f index_tts_api.py
!pkill -f cloudflared
# 额外清理可能占用的 7890 端口
!fuser -k 7890/tcp > /dev/null 2>&1

# 3. 下载 Cloudflare 穿透工具 (如果不存在)
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("⬇️ 正在安装 Cloudflare...")
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

# 4. 后台启动 API
# 修正点：移除了报错的 --port 参数，让它使用默认配置
print("🚀 正在启动 IndexTTS API (后台运行)...")
log_file = open("api_log.txt", "w")

api_process = subprocess.Popen(
    ["uv", "run", "index_tts_api.py", "--model_dir", "/content/index-tts/checkpoints/IndexTTS-2"],
    stdout=log_file,
    stderr=log_file
)

# 5. 等待 API 就绪
# 修正点：根据日志，服务实际运行在 7890 端口
detected_port = 7890
print(f"⏳ 等待 API 启动 (预计 20 秒，目标端口 {detected_port})...")
time.sleep(20)

# 6. 启动穿透
print(f"🔗 正在建立 Cloudflare 隧道 (连接 127.0.0.1:{detected_port})...")
tunnel_process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{detected_port}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# 7. 获取并打印地址
time.sleep(5)
found_url = False
retry_count = 0

while not found_url and retry_count < 15:
    # 逐行读取输出
    line = tunnel_process.stderr.readline().decode('utf-8')
    if not line:
        time.sleep(1)
        retry_count += 1
        continue

    if 'trycloudflare.com' in line:
        url_match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if url_match:
            public_url = url_match.group(0)
            print("\n" + "="*50)
            print(f"✅ 成功！API 服务已建立 (内部端口 {detected_port})")
            print(f"📋 请在软件中填入以下地址 (不要加后缀):")
            print(f"\n👉 {public_url} \n")
            print("="*50)
            found_url = True

if not found_url:
    print("❌ 获取地址超时。请检查下方的 api_log.txt 是否有报错。")
    # 打印最后几行日志帮助排查
    print("\n--- api_log.txt (Last 10 lines) ---")
    !tail -n 10 api_log.txt

In [ ]:
!cat api_log.txt